In [2]:
import datetime
import pandas as pd
import requests
import os
import sys
import json
from threading import Thread
# # uppend upper dir
# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# with open(upper_dir + "/trade_core_config.json") as f:
#     config = json.load(f)

with open("/home/trade_core/trade_core_config.json") as f:
    config = json.load(f)

node = config['node']
prod = config['node_settings'][node]['prod']

class AcwApi:
    def __init__(self, prod=prod):
        if prod is False:
            self.verify = False
            self.url = config['acw_setting']['dev_url']
        else:
            self.verify = True
            self.url = config['acw_setting']['prod_url']
        self.message_url = "messagecore/messages/"
        self.node_url = "tradecore/nodes/"

    def get_message(self, id=None):
        url = self.url + self.message_url
        if id is not None:
            fetch_list = False
            response = requests.get(url + str(id) + "/", verify=self.verify)
        else:
            fetch_list = True
            response = requests.get(url, verify=self.verify)
        if response.status_code == 200:
            if fetch_list:
                return pd.DataFrame(response.json()["results"])
            else:
                return pd.DataFrame([response.json()])
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def create_message(self, telegram_chat_id, title, origin, type, content=None, remark=None, code=None, sent=False, send_times=1, send_term=1):
        url = self.url + self.message_url
        new_message_dict = {
            "datetime": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S"),
            "telegram_chat_id": telegram_chat_id,
            "title": title,
            "origin": origin,
            "type": type,
            "content": content,
            "remark": remark,
            "code": code,
            "sent": sent,
            "send_times": send_times,
            "send_term": send_term
        }
        response = requests.post(url, json=new_message_dict, verify=self.verify)
        if response.status_code == 201:
            return response.json()
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def create_message_thread(self, telegram_chat_id, title, origin, type, content=None, remark=None, code=None, sent=False, send_times=1, send_term=1):
        t = Thread(target=self.create_message, args=(telegram_chat_id, title, origin, type, content, remark, code, sent, send_times, send_term))
        t.start()

    def delete_message(self, id):
        url = self.url + self.message_url
        response = requests.delete(url + str(id) + "/", verify=self.verify)
        if response.status_code == 204:
            return True
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def get_node(self, id=None):
        url = self.url + self.node_url
        if id is not None:
            fetch_list = False
            response = requests.get(url + str(id) + "/", verify=self.verify)
        else:
            fetch_list = True
            response = requests.get(url, verify=self.verify)
        if response.status_code == 200:
            if fetch_list:
                return pd.DataFrame(response.json()["results"])
            else:
                return pd.DataFrame([response.json()])
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)

In [4]:
import time

In [3]:
acw_api = AcwApi(prod=False)

In [14]:
start = time.time()
acw_api.create_message(telegram_chat_id=1390695186, title="test", origin="test", type="info", content="test", remark="test", code=None, sent=False, send_times=1, send_term=1)
print(time.time()-start)

0.10544180870056152


/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [15]:
start = time.time()
acw_api.create_message_thread(telegram_chat_id=1390695186, title="test", origin="test", type="info", content="test", remark="test", code=None, sent=False, send_times=1, send_term=1)
print(time.time()-start)

0.0018434524536132812


/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [12]:
response.json()

{'results': [{'uuid': '342b6270-cfd9-4676-90e4-125f67e4c03c',
   'email': 'superuser@halo-soft.net',
   'username': 'superuser',
   'first_name': 'Super',
   'last_name': 'User',
   'role': 'internal',
   'is_active': True,
   'date_joined': '2023-10-04T04:18:11+0000',
   'profile': {'referral': None,
    'level': 1,
    'points': 0,
    'picture': None,
    'user': '342b6270-cfd9-4676-90e4-125f67e4c03c'},
   'favorite_assets': [],
   'socialapps': [],
   'telegram_chat_id': None,
   'trade_config_allocations': []},
  {'uuid': '2c280bd2-6289-4e24-b1ee-5b7c5786265e',
   'email': 'ceo@halo-soft.net',
   'username': '헤일로테스트',
   'first_name': 'Chang-un',
   'last_name': 'Park',
   'role': 'user',
   'is_active': True,
   'date_joined': '2023-10-20T02:23:59+0000',
   'profile': {'referral': None,
    'level': 1,
    'points': 0,
    'picture': 'https://lh3.googleusercontent.com/a/ACg8ocIerr63_JO6LcR_Y4tWhz6pDV69zj8hMuvcc2eM_4fQ=s96-c',
    'user': '2c280bd2-6289-4e24-b1ee-5b7c5786265e'},
 

In [17]:
acw_api.get_node()

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-dev.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,id,name,url,description,max_user_count,market_code_services
0,1,trade_core_dev,http://221.148.128.205:20081,changun's test tradecore server,300,"[UPBIT_SPOT/KRW:BINANCE_USD_M/USDT, BITHUMB_SP..."


In [34]:
node = 'trade_core_dev'

all_node_df = acw_api.get_node()
node_df = all_node_df[all_node_df['name']==node]
if len(node_df) != 1:
    raise Exception(f"Node not found or not unique, len(node_df)={len(node_df)}")
node_df['market_code_services'].values[0]



/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-dev.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


['UPBIT_SPOT/KRW:BINANCE_USD_M/USDT', 'BITHUMB_SPOT/KRW:BINANCE_SPOT/USDT']